# Manuscript Figure 5

This is a public Colab adaptation of `src/06_two_dates/06b_figure2_fast_counterfactual_colab.ipynb` from the team snapshot. It uses the original figure and inference cells with paths set to the archived data. Fitted models and cell outputs are not published. Start with a fresh CPU runtime and run from the top.


## Load the team data

Choose the full team ZIP (default), or set `USE_EXISTING_DRIVE_RASTERS = True` to use the small map-support ZIP plus existing Drive folders. Set `ARCHIVE_PATH` if the ZIP is in a subfolder. Both routes use verified inputs and neither signs into Earth Engine. Save generated results to Drive before disconnecting.


In [ ]:
import hashlib
import os
import tempfile
import zipfile
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

USE_EXISTING_DRIVE_RASTERS = False
MY_DRIVE = Path("/content/drive/MyDrive")
ARCHIVE_PATH = MY_DRIVE / (
    "Water_Quality_Map_Support.zip" if USE_EXISTING_DRIVE_RASTERS else "Water_Quality_Processed_Handoff.zip"
)
EXPECTED_SHA256 = "917042102ff89628664e459628e6c7719928e3256a3d17c3e3ca9e51edce480a" if USE_EXISTING_DRIVE_RASTERS else "1f1039b7a6d113685c2c95f840f2a475be5e936e02090bcf5283f2ba9b506773"


HANDOFF_ROOT = Path("/content/wqr-handoff-" + EXPECTED_SHA256[:12])
marker = HANDOFF_ROOT / ".verified-archive-sha256"

if not (marker.is_file() and marker.read_text().strip() == EXPECTED_SHA256):
    if not ARCHIVE_PATH.is_file():
        raise FileNotFoundError(
            f"{ARCHIVE_PATH} was not found. Ask the project owner for the selected ZIP "
            "and point ARCHIVE_PATH to its location on the mounted Drive."
        )
    with tempfile.TemporaryDirectory(dir=HANDOFF_ROOT.parent) as tmp:
        tmp = Path(tmp)
        local_zip = tmp / ARCHIVE_PATH.name
        digest = hashlib.sha256()
        with ARCHIVE_PATH.open("rb") as src, local_zip.open("wb") as dst:
            for chunk in iter(lambda: src.read(8 * 1024 * 1024), b""):
                digest.update(chunk)
                dst.write(chunk)
        if digest.hexdigest() != EXPECTED_SHA256:
            raise ValueError("This ZIP differs from the verified snapshot. Check the filename and download.")
        extracted = tmp / "snapshot"
        with zipfile.ZipFile(local_zip) as archive:
            archive.extractall(extracted)
        (extracted / marker.name).write_text(EXPECTED_SHA256 + "\n")
        os.replace(extracted, HANDOFF_ROOT)

print("Team data ready:", HANDOFF_ROOT)


In [ ]:
# Only needed for the optional direct-Drive route. Edit paths for Drive shortcuts.
import csv

raster_folders = {'two_dates/features': 'Water_Quality_RS_Figure2_Features', 'two_dates/rgb': 'Water_Quality_RS_Figure2_RGB'}
expected_prefixes = ['WB02_20240604', 'WB02_20240619']
DRIVE_RASTER_PATHS = {group: MY_DRIVE / name for group, name in raster_folders.items()}

if USE_EXISTING_DRIVE_RASTERS:
    with (HANDOFF_ROOT / "RASTER_MANIFEST.csv").open(newline="", encoding="utf-8") as handle:
        raster_manifest = list(csv.DictReader(handle))
    for group, drive_folder in DRIVE_RASTER_PATHS.items():
        for prefix in expected_prefixes:
            suffix = "_sentinel2_rgb.tif" if group.endswith("/rgb") else "_moving_window_features.tif"
            relative = f"data_ready/rasters/{group}/{prefix}{suffix}"
            matches = [row for row in raster_manifest if row["path"] == relative]
            if len(matches) != 1:
                raise ValueError(f"No unique reference file for {relative}")
            row = matches[0]
            destination = HANDOFF_ROOT / relative
            if destination.is_file() and destination.stat().st_size == int(row["bytes"]):
                continue  # Only verified files are moved to this runtime-local folder.
            original_raster = drive_folder / destination.name
            if not original_raster.is_file():
                raise FileNotFoundError(f"{original_raster} not found. Check DRIVE_RASTER_PATHS.")
            destination.parent.mkdir(parents=True, exist_ok=True)
            temporary = destination.with_suffix(".part")
            try:
                digest = hashlib.sha256()
                with original_raster.open("rb") as src, temporary.open("wb") as dst:
                    for chunk in iter(lambda: src.read(8 * 1024 * 1024), b""):
                        digest.update(chunk)
                        dst.write(chunk)
                if digest.hexdigest() != row["sha256"]:
                    raise ValueError(f"{original_raster.name} differs from the paper snapshot.")
                temporary.replace(destination)
            finally:
                temporary.unlink(missing_ok=True)
    print("Verified selected Drive rasters:", ", ".join(expected_prefixes))


# Figure 2: Fast Contrasting-Date Prediction and Two-Block Decomposition

This Google Colab notebook compares the two independently selected
Sentinel-2 dates for **Thale Luang (WB02)** without computing all-pixel
SHAP. It retains the locked S2-plus-context models, original target units,
strict common valid low-OOD water pixels, and the independently selected
dates.

The temporal prediction change is decomposed into two model-input blocks:

- **Satellite/scene block:** Sentinel-2 bands, indices, and scene-dependent
  spectral features.
- **Model-context block:** every remaining predictor used by the locked
  model, including hydrometeorology, seasonality, water/QA variables, and
  static landscape or historical-water variables where applicable.

For each pixel, four model configurations are evaluated:
`f(S1,C1)`, `f(S1,C2)`, `f(S2,C1)`, and `f(S2,C2)`. Averaging both swap
orders gives an exact two-block Shapley-style decomposition of the modeled
date-to-date prediction difference.

This is a **model-based sensitivity decomposition**, not a causal effect
estimate and not same-date field validation. DO remains an indirect
**DO proxy**.


In [ ]:
# Colab/runtime setup. Install before importing fitted-model dependencies.
from pathlib import Path
import importlib.metadata
import importlib.util
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None
required_versions = {
    "scikit-learn": "1.7.2",
    "catboost": "1.2.10",
}
install_specs = []
for package, version in required_versions.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        install_specs.append(f"{package}=={version}")

for module_name, package_name in {
    "rasterio": "rasterio",
    "joblib": "joblib",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
}.items():
    if importlib.util.find_spec(module_name) is None:
        install_specs.append(package_name)

if install_specs:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *install_specs
    ])

import base64
import hashlib
import json
import math
import time
import warnings

import joblib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize, TwoSlopeNorm
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MaxNLocator, FuncFormatter
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.windows import Window
from scipy.stats import spearmanr
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

print("Runtime:", "Google Colab" if IN_COLAB else "local")
print("scikit-learn:", importlib.metadata.version("scikit-learn"))
print("catboost:", importlib.metadata.version("catboost"))
print("rasterio:", rasterio.__version__)


## 1. Configuration


In [ ]:
# Edit only the override values when your Drive layout differs.
MODEL_DATA_PATH_OVERRIDE = HANDOFF_ROOT / "data_ready/model/rswq_model_input_relaxed_s2_context_2562_2568.csv"
PREPARATION_DIR_OVERRIDE = HANDOFF_ROOT / "data_ready/maps/two_date_preparation"
FEATURE_DIR_OVERRIDE = HANDOFF_ROOT / "data_ready/rasters/two_dates/features"
RGB_DIR_OVERRIDE = HANDOFF_ROOT / "data_ready/rasters/two_dates/rgb"
OUTPUT_DIR_OVERRIDE = HANDOFF_ROOT / "run_output/colab_fig05"

TARGET_MAP_ID = "WB02"
TARGET_WATERBODY = "Thale Luang"
EXPECTED_DATES = ["2024-06-04", "2024-06-19"]

MAX_OOD_FEATURE_FRACTION = 0.25
BLOCK_SIZE = 512
RANDOM_STATE = 20260810
NODATA = -9999.0
OVERWRITE_PREDICTIONS = False
OVERWRITE_COUNTERFACTUAL = False
PREVIEW_MAX_DIMENSION = 1100

MODEL_DATA_FILENAME = (
    "rswq_model_input_relaxed_s2_context_2562_2568.csv"
)

PREPARATION_DIR = PREPARATION_DIR_OVERRIDE
FEATURE_DIR = FEATURE_DIR_OVERRIDE
RGB_DIR = RGB_DIR_OVERRIDE
OUTPUT_DIR = OUTPUT_DIR_OVERRIDE
OUTPUT_ROOT = OUTPUT_DIR  # Used by the embedded locked-artifact cell.

RASTER_DIR = OUTPUT_DIR / "rasters"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
for directory in [
    OUTPUT_DIR, RASTER_DIR, TABLE_DIR, FIGURE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Preparation:", PREPARATION_DIR)
print("Feature exports:", FEATURE_DIR)
print("RGB exports:", RGB_DIR)
print("Output:", OUTPUT_DIR)


## 2. Reconstruct and verify the locked model artifacts


In [ ]:
# Load the exact historical setup cell from the verified team snapshot.
import hashlib
import json

original = HANDOFF_ROOT / "src/06_two_dates/06b_figure2_fast_counterfactual_colab.ipynb"
private_notebook = json.loads(original.read_text(encoding="utf-8-sig"))
private_setup = "".join(private_notebook["cells"][5]["source"])
if hashlib.sha256(private_setup.encode()).hexdigest() != "4ba78717c825c99ad9a25ee378a610364fd8431f4315fb624e1e6482b34e52b1":
    raise ValueError("The locked-model setup cell does not match the team snapshot.")
exec(compile(private_setup, str(original), "exec"))
del private_setup, private_notebook


In [ ]:
TARGET_CONFIG = {
    "Turbidity": {
        "artifact_id": "turbidity_s2_context",
        "file": "turbidity_s2_context_linearsvr.joblib",
        "target_col": "turbidity_NTU_clean",
        "unit": "NTU",
        "method": "Two-block counterfactual decomposition in NTU",
    },
    "SS": {
        "artifact_id": "ss_s2_context",
        "file": "ss_s2_context_two_stage_bundle.joblib",
        "target_col": "SS_mg_l_clean",
        "unit": "mg/L",
        "method": "Two-block counterfactual decomposition of the soft-gated prediction in mg/L",
    },
    "DO proxy": {
        "artifact_id": "do_s2_context",
        "file": "do_s2_context_catboost.joblib",
        "target_col": "DO_mg_l_clean",
        "unit": "mg/L",
        "method": "Two-block counterfactual decomposition in mg/L",
    },
}
TARGET_ORDER = list(TARGET_CONFIG)

bundles = {}
for target, config in TARGET_CONFIG.items():
    bundle = joblib.load(ARTIFACT_ROOT / config["file"])
    if bundle["artifact_id"] != config["artifact_id"]:
        raise AssertionError(
            f"{target}: artifact ID mismatch: {bundle['artifact_id']}"
        )
    bundles[target] = bundle

selected_artifact_ids = {config["artifact_id"] for config in TARGET_CONFIG.values()}
training_ranges = training_ranges.loc[
    training_ranges["artifact_id"].isin(selected_artifact_ids)
].copy()

inventory_rows = []
for target, bundle in bundles.items():
    metrics = bundle["validation_metrics"]
    inventory_rows.append({
        "target": target,
        "artifact_id": bundle["artifact_id"],
        "strategy": bundle["prediction_contract"]["strategy"],
        "feature_set": bundle["feature_set"],
        "n_features": len(bundle["feature_names"]),
        "n_train": bundle["n_train"],
        "n_test": bundle["n_test"],
        "r2": metrics.get("original_r2", metrics.get("r2")),
        "rmse": metrics.get("original_rmse", metrics.get("rmse")),
        "unit": TARGET_CONFIG[target]["unit"],
        "decomposition_method": TARGET_CONFIG[target]["method"],
    })
model_inventory = pd.DataFrame(inventory_rows)
model_inventory.to_csv(
    TABLE_DIR / "figure2_locked_model_inventory.csv",
    index=False, encoding="utf-8-sig",
)
display(model_inventory.round(4))


## 3. Load the original model dataset and reproduce the locked training splits


In [ ]:
def normalized_difference(a, b, eps=1e-12):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    denominator = a + b
    values = np.where(np.abs(denominator) > eps, (a - b) / denominator, np.nan)
    return np.where(np.isfinite(values), values, np.nan)


def safe_ratio(a, b, eps=1e-12):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    values = np.where(np.abs(b) > eps, a / b, np.nan)
    return np.where(np.isfinite(values), values, np.nan)


def add_custom_s2_features(data):
    out = data.copy()
    blue, green, red = out["B2_median"], out["B3_median"], out["B4_median"]
    nir, swir1, swir2 = out["B8_median"], out["B11_median"], out["B12_median"]
    out["NDWI1"] = normalized_difference(green, nir)
    out["NDWI2"] = normalized_difference(nir, swir1)
    out["NDWI3"] = normalized_difference(nir, swir2)
    out["B4_B3_ratio"] = safe_ratio(red, green)
    out["B8_B4_ratio"] = safe_ratio(nir, red)
    out["NIR_Blue_ratio"] = safe_ratio(nir, blue)
    out["NIR_Green_ratio"] = safe_ratio(nir, green)
    out["NIR_Red_ratio"] = safe_ratio(nir, red)
    out["Red_NIR_ND"] = normalized_difference(red, nir)
    return out


def resolve_model_data():
    path = Path(MODEL_DATA_PATH_OVERRIDE)
    if not path.is_file():
        raise FileNotFoundError(path)
    return path


MODEL_DATA_PATH = resolve_model_data()
model_data = add_custom_s2_features(pd.read_csv(MODEL_DATA_PATH, low_memory=False))
model_data["year_be"] = pd.to_numeric(model_data["year_be"], errors="coerce")
model_data["abs_date_diff_days"] = pd.to_numeric(
    model_data["abs_date_diff_days"], errors="coerce"
)

train_splits = {}
split_rows = []
for target, config in TARGET_CONFIG.items():
    bundle = bundles[target]
    features = list(bundle["feature_names"])
    missing = [feature for feature in features if feature not in model_data.columns]
    if missing:
        raise KeyError(f"{target}: model dataset is missing {missing}")

    target_values = pd.to_numeric(model_data[config["target_col"]], errors="coerce")
    mask = target_values.notna() & model_data["abs_date_diff_days"].le(1)
    if target == "SS":
        mask &= target_values.le(float(bundle["ss_max_mg_l"]))
    train = model_data.loc[
        mask & model_data["year_be"].between(2562, 2566)
    ].copy()
    evaluation = model_data.loc[
        mask & model_data["year_be"].between(2567, 2568)
    ].copy()
    if len(train) != int(bundle["n_train"]) or len(evaluation) != int(bundle["n_test"]):
        raise AssertionError(
            f"{target}: split mismatch {len(train)}/{len(evaluation)}; "
            f"artifact expects {bundle['n_train']}/{bundle['n_test']}"
        )
    train_splits[target] = train
    split_rows.append({
        "target": target,
        "n_train": len(train),
        "n_evaluation": len(evaluation),
        "train_years_be": "2562-2566",
        "evaluation_years_be": "2567-2568",
        "within_days": 1,
        "ss_max_mg_l": bundle.get("ss_max_mg_l", np.nan),
    })

split_audit = pd.DataFrame(split_rows)
split_audit.to_csv(
    TABLE_DIR / "figure2_training_split_audit.csv",
    index=False, encoding="utf-8-sig",
)
print("Model data:", MODEL_DATA_PATH)
display(split_audit)


## 4. Resolve and validate the two-date exports


In [ ]:
def resolve_one_csv(directory, file_name):
    path = Path(directory) / file_name
    if path.exists():
        return path
    matches = list(Path(directory).rglob(file_name)) if Path(directory).exists() else []
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one {file_name}; found {matches}")
    return matches[0]


def resolve_job_raster(directory, job_id, marker):
    directory = Path(directory)
    if not directory.exists():
        raise FileNotFoundError(directory)
    matches = sorted({
        path for path in directory.rglob("*.tif")
        if job_id in path.name and marker in path.name
    })
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one {marker} GeoTIFF for {job_id}; found {len(matches)}:\n"
            + "\n".join(str(path) for path in matches)
        )
    return matches[0]


SELECTED_MANIFEST_PATH = resolve_one_csv(
    PREPARATION_DIR, "figure2_selected_date_pair_manifest.csv"
)
CONTEXT_AUDIT_PATH = resolve_one_csv(
    PREPARATION_DIR, "figure2_candidate_date_context_audit.csv"
)
selected_manifest = pd.read_csv(SELECTED_MANIFEST_PATH, low_memory=False)
selected_manifest["sat_date"] = pd.to_datetime(
    selected_manifest["sat_date"], errors="coerce"
).dt.strftime("%Y-%m-%d")
selected_manifest = selected_manifest.sort_values("sat_date").reset_index(drop=True)

if selected_manifest["sat_date"].tolist() != EXPECTED_DATES:
    raise AssertionError(
        f"Unexpected selected dates: {selected_manifest['sat_date'].tolist()}"
    )
if selected_manifest["selection_used_predictions"].astype(str).str.lower().isin(["true", "1"]).any():
    raise AssertionError("Date selection must remain independent of predictions.")

file_rows = []
for row in selected_manifest.to_dict("records"):
    job_id = row["job_id"]
    feature_path = resolve_job_raster(FEATURE_DIR, job_id, "moving_window_features")
    rgb_path = resolve_job_raster(RGB_DIR, job_id, "sentinel2_rgb")
    file_rows.append({**row, "feature_path": feature_path, "rgb_path": rgb_path})
jobs = pd.DataFrame(file_rows)

reference_signature = None
grid_rows = []
required_features = sorted({
    feature for bundle in bundles.values() for feature in bundle["feature_names"]
} | {"prediction_support"})
for row in jobs.to_dict("records"):
    with rasterio.open(row["feature_path"]) as feature_src, rasterio.open(row["rgb_path"]) as rgb_src:
        signature = (
            feature_src.width, feature_src.height, feature_src.crs,
            tuple(feature_src.transform),
        )
        if reference_signature is None:
            reference_signature = signature
        feature_names = list(feature_src.descriptions)
        missing = [name for name in required_features if name not in feature_names]
        rgb_signature = (
            rgb_src.width, rgb_src.height, rgb_src.crs,
            tuple(rgb_src.transform),
        )
        grid_rows.append({
            "job_id": row["job_id"],
            "sat_date": row["sat_date"],
            "feature_bands": feature_src.count,
            "rgb_bands": rgb_src.count,
            "width": feature_src.width,
            "height": feature_src.height,
            "crs": str(feature_src.crs),
            "feature_grid_matches_reference": signature == reference_signature,
            "rgb_grid_matches_feature": rgb_signature == signature,
            "model_features_complete": not missing,
            "missing_features": ", ".join(missing),
        })

grid_audit = pd.DataFrame(grid_rows)
if not grid_audit[[
    "feature_grid_matches_reference", "rgb_grid_matches_feature",
    "model_features_complete",
]].all().all():
    raise AssertionError("Two-date raster validation failed.")
grid_audit.to_csv(
    TABLE_DIR / "figure2_input_grid_audit.csv",
    index=False, encoding="utf-8-sig",
)
display(jobs[["job_id", "sat_date", "feature_path", "rgb_path"]])
display(grid_audit)


## 5. Locked block-wise prediction for both observed dates


In [ ]:
PREDICTION_BANDS = [
    "prediction_support",
    "turbidity_s2_context", "ss_s2_context", "do_s2_context",
    "turbidity_s2_context_ood_fraction",
    "ss_s2_context_ood_fraction",
    "do_s2_context_ood_fraction",
]
PREDICTION_BAND_INDEX = {
    name: index + 1 for index, name in enumerate(PREDICTION_BANDS)
}
OUTPUT_NAME_BY_TARGET = {
    "Turbidity": "turbidity_s2_context",
    "SS": "ss_s2_context",
    "DO proxy": "do_s2_context",
}

range_lookup = {}
for target, config in TARGET_CONFIG.items():
    artifact_id = config["artifact_id"]
    indexed = training_ranges.loc[
        training_ranges["artifact_id"].eq(artifact_id)
    ].set_index("feature_name")
    features = bundles[target]["feature_names"]
    range_lookup[target] = {
        "minimum": indexed.loc[features, "training_min"].to_numpy(float),
        "maximum": indexed.loc[features, "training_max"].to_numpy(float),
    }


def exact_prediction(bundle, frame):
    features = list(bundle["feature_names"])
    frame = frame.loc[:, features]
    strategy = bundle["prediction_contract"]["strategy"]
    if strategy == "log1p_regression":
        output = np.asarray(bundle["estimator"].predict(frame), dtype=float)
        original = np.expm1(np.clip(output, 0.0, None))
    elif strategy == "two_stage_soft_gate":
        probabilities = np.asarray(bundle["classifier"].predict_proba(frame), dtype=float)
        classes = np.asarray(bundle["classifier"].classes_)
        high_column = np.flatnonzero(classes == 1)
        if high_column.size != 1:
            raise ValueError("Could not identify the SS high class.")
        p_high = probabilities[:, high_column[0]]
        high_log = np.asarray(bundle["regressor"].predict(frame), dtype=float)
        high_original = np.expm1(np.clip(high_log, 0.0, None))
        original = p_high * high_original + (1.0 - p_high) * float(bundle["low_value"])
        output = original.copy()
    elif strategy == "raw_regression":
        original = np.asarray(bundle["estimator"].predict(frame), dtype=float)
        output = original.copy()
    else:
        raise ValueError(f"Unsupported strategy: {strategy}")
    return output.reshape(-1), original.reshape(-1)


def ood_fraction(target, matrix):
    bounds = range_lookup[target]
    finite = np.isfinite(matrix)
    outside = finite & (
        (matrix < bounds["minimum"][None, :])
        | (matrix > bounds["maximum"][None, :])
    )
    return outside.sum(axis=1) / matrix.shape[1]


def iter_windows(width, height, block_size):
    for row_off in range(0, height, block_size):
        for col_off in range(0, width, block_size):
            yield Window(
                col_off, row_off,
                min(block_size, width - col_off),
                min(block_size, height - row_off),
            )


def prediction_path(job_id):
    return RASTER_DIR / f"{job_id}_figure2_predictions.tif"


def run_prediction_job(job):
    output_path = prediction_path(job["job_id"])
    if output_path.exists() and not OVERWRITE_PREDICTIONS:
        print("Using cached prediction:", output_path)
        return output_path

    feature_path = Path(job["feature_path"])
    with rasterio.open(feature_path) as source:
        band_map = {
            name: index for index, name in enumerate(source.descriptions, start=1)
            if name
        }
        union_features = []
        for target in TARGET_ORDER:
            for feature in bundles[target]["feature_names"]:
                if feature not in union_features:
                    union_features.append(feature)
        read_names = union_features + ["prediction_support"]
        read_indexes = [band_map[name] for name in read_names]

        profile = source.profile.copy()
        profile.update(
            driver="GTiff", count=len(PREDICTION_BANDS), dtype="float32",
            nodata=NODATA, compress="deflate", predictor=3,
            BIGTIFF="IF_SAFER",
        )
        windows = list(iter_windows(source.width, source.height, BLOCK_SIZE))
        with rasterio.open(output_path, "w", **profile) as destination:
            for index, name in enumerate(PREDICTION_BANDS, start=1):
                destination.set_band_description(index, name)
            destination.update_tags(
                workflow="figure2_contrasting_dates",
                model_configuration="locked_s2_plus_context",
                job_id=job["job_id"], sat_date=job["sat_date"],
            )

            for window_number, window in enumerate(windows, start=1):
                raw = source.read(read_indexes, window=window, masked=True)
                raw = np.ma.asarray(raw).filled(np.nan).astype(float)
                block = {name: raw[index] for index, name in enumerate(read_names)}
                support = block["prediction_support"]
                selected = np.flatnonzero(np.isfinite(support.reshape(-1)) & (support.reshape(-1) > 0))
                shape = (len(PREDICTION_BANDS), int(window.height), int(window.width))
                output = np.full(shape, NODATA, dtype=np.float32)
                support_output = output[PREDICTION_BAND_INDEX["prediction_support"] - 1].reshape(-1)
                support_output[selected] = support.reshape(-1)[selected].astype(np.float32)

                if selected.size:
                    for target in TARGET_ORDER:
                        bundle = bundles[target]
                        features = list(bundle["feature_names"])
                        matrix = np.column_stack([
                            block[name].reshape(-1)[selected] for name in features
                        ])
                        frame = pd.DataFrame(matrix, columns=features)
                        _, values = exact_prediction(bundle, frame)
                        values = np.clip(values, 0.0, None)
                        output_name = OUTPUT_NAME_BY_TARGET[target]
                        target_output = output[PREDICTION_BAND_INDEX[output_name] - 1].reshape(-1)
                        target_output[selected] = values.astype(np.float32)
                        ood_name = f"{output_name}_ood_fraction"
                        ood_output = output[PREDICTION_BAND_INDEX[ood_name] - 1].reshape(-1)
                        ood_output[selected] = ood_fraction(target, matrix).astype(np.float32)

                destination.write(output, window=window)
                if window_number == 1 or window_number % 10 == 0 or window_number == len(windows):
                    print(f"{job['job_id']}: window {window_number}/{len(windows)}")
    print("Saved:", output_path)
    return output_path


prediction_paths = {}
for job in jobs.to_dict("records"):
    prediction_paths[job["job_id"]] = run_prediction_job(job)


## 6. Strict common low-OOD water mask and observed prediction change


In [ ]:
def band_map(source):
    return {
        name: index for index, name in enumerate(source.descriptions, start=1)
        if name
    }


def read_named(source, name):
    index = band_map(source)[name]
    values = source.read(index).astype(float)
    if source.nodata is not None:
        values[values == source.nodata] = np.nan
    return values


date_jobs = jobs.sort_values("sat_date").to_dict("records")
prediction_arrays = {}
support_arrays = {}
ood_arrays = {}
target_date_masks = {target: {} for target in TARGET_ORDER}

for job in date_jobs:
    job_id = job["job_id"]
    with rasterio.open(prediction_paths[job_id]) as source:
        support = read_named(source, "prediction_support")
        support_arrays[job_id] = support
        prediction_arrays[job_id] = {}
        ood_arrays[job_id] = {}
        for target in TARGET_ORDER:
            output_name = OUTPUT_NAME_BY_TARGET[target]
            prediction = read_named(source, output_name)
            ood = read_named(source, f"{output_name}_ood_fraction")
            prediction_arrays[job_id][target] = prediction
            ood_arrays[job_id][target] = ood
            target_date_masks[target][job_id] = (
                np.isfinite(support) & (support > 0)
                & np.isfinite(prediction)
                & np.isfinite(ood)
                & (ood <= MAX_OOD_FEATURE_FRACTION)
            )

target_common_masks = {}
for target in TARGET_ORDER:
    target_common_masks[target] = np.logical_and.reduce([
        target_date_masks[target][job["job_id"]] for job in date_jobs
    ])
strict_common_mask = np.logical_and.reduce([
    target_common_masks[target] for target in TARGET_ORDER
])
if strict_common_mask.sum() == 0:
    raise RuntimeError("No strict common low-OOD water pixels remain.")

reference_path = Path(date_jobs[0]["feature_path"])
with rasterio.open(reference_path) as reference:
    mask_profile = reference.profile.copy()
    mask_profile.update(
        driver="GTiff", count=1, dtype="uint8", nodata=0,
        compress="deflate", predictor=2,
    )
    common_mask_path = RASTER_DIR / "WB02_20240604_20240619_strict_common_mask.tif"
    with rasterio.open(common_mask_path, "w", **mask_profile) as destination:
        destination.write(strict_common_mask.astype(np.uint8), 1)
        destination.set_band_description(1, "strict_common_low_ood_water_mask")

mask_rows = []
for target in TARGET_ORDER:
    date_counts = [
        int(target_date_masks[target][job["job_id"]].sum()) for job in date_jobs
    ]
    common_count = int(target_common_masks[target].sum())
    mask_rows.append({
        "target": target,
        "valid_date_1": date_counts[0],
        "valid_date_2": date_counts[1],
        "target_common_pixels": common_count,
        "strict_three_target_common_pixels": int(strict_common_mask.sum()),
        "strict_retained_pct_of_target_common": 100 * strict_common_mask.sum() / common_count,
    })
common_mask_audit = pd.DataFrame(mask_rows)
common_mask_audit.to_csv(
    TABLE_DIR / "figure2_common_mask_audit.csv",
    index=False, encoding="utf-8-sig",
)

prediction_rows = []
delta_pixel_frames = {}
job1, job2 = date_jobs[0]["job_id"], date_jobs[1]["job_id"]
for target in TARGET_ORDER:
    values1 = prediction_arrays[job1][target][strict_common_mask]
    values2 = prediction_arrays[job2][target][strict_common_mask]
    delta = values2 - values1
    delta_pixel_frames[target] = pd.DataFrame({
        "prediction_date_1": values1,
        "prediction_date_2": values2,
        "prediction_delta": delta,
    })
    prediction_rows.append({
        "target": target,
        "unit": TARGET_CONFIG[target]["unit"],
        "date_1": date_jobs[0]["sat_date"],
        "date_2": date_jobs[1]["sat_date"],
        "n_common_pixels": len(delta),
        "date_1_median": np.median(values1),
        "date_2_median": np.median(values2),
        "delta_mean": np.mean(delta),
        "delta_median": np.median(delta),
        "delta_q25": np.quantile(delta, 0.25),
        "delta_q75": np.quantile(delta, 0.75),
        "positive_pixel_pct": 100 * np.mean(delta > 0),
        "negative_pixel_pct": 100 * np.mean(delta < 0),
    })
prediction_change_summary = pd.DataFrame(prediction_rows)
prediction_change_summary.to_csv(
    TABLE_DIR / "figure2_prediction_change_summary.csv",
    index=False, encoding="utf-8-sig",
)
display(common_mask_audit.round(2))
display(prediction_change_summary.round(4))
print("Common mask:", common_mask_path)


## 7. Environmental-context contrast


In [ ]:
DYNAMIC_CONTEXT_FEATURES = [
    "rain_gsmap_prior_24h_mm",
    "rain_gsmap_prior_336h_mm",
    "rain_gsmap_prior_720h_mm",
    "rainy_hours_gsmap_prior_336h",
    "era5h_runoff_prior_1h_mm",
    "era5h_runoff_prior_24h_mm",
    "era5h_air_temp_prior_72h_C",
    "era5h_wind_max_prior_168h_m_s",
    "era5h_solar_prior_168h_MJ_m2",
]
CONTEXT_UNITS = {
    "rain_gsmap_prior_24h_mm": "mm",
    "rain_gsmap_prior_336h_mm": "mm",
    "rain_gsmap_prior_720h_mm": "mm",
    "rainy_hours_gsmap_prior_336h": "hours",
    "era5h_runoff_prior_1h_mm": "mm",
    "era5h_runoff_prior_24h_mm": "mm",
    "era5h_air_temp_prior_72h_C": "deg C",
    "era5h_wind_max_prior_168h_m_s": "m/s",
    "era5h_solar_prior_168h_MJ_m2": "MJ/m2",
}
CONTEXT_SHORT_LABELS = {
    "rain_gsmap_prior_24h_mm": "Rainfall, prior 24 h",
    "rain_gsmap_prior_336h_mm": "Rainfall, prior 336 h",
    "rain_gsmap_prior_720h_mm": "Rainfall, prior 720 h",
    "rainy_hours_gsmap_prior_336h": "Rainy hours, prior 336 h",
    "era5h_runoff_prior_1h_mm": "Runoff, prior 1 h",
    "era5h_runoff_prior_24h_mm": "Runoff, prior 24 h",
    "era5h_air_temp_prior_72h_C": "Air temperature, prior 72 h",
    "era5h_wind_max_prior_168h_m_s": "Maximum wind, prior 168 h",
    "era5h_solar_prior_168h_MJ_m2": "Solar radiation, prior 168 h",
}
CONTEXT_GROUP = {
    **{name: "Rainfall" for name in DYNAMIC_CONTEXT_FEATURES[:4]},
    **{name: "Runoff" for name in DYNAMIC_CONTEXT_FEATURES[4:6]},
    "era5h_air_temp_prior_72h_C": "Air temperature",
    "era5h_wind_max_prior_168h_m_s": "Wind",
    "era5h_solar_prior_168h_MJ_m2": "Solar radiation",
}
SHAP_DYNAMIC_WEIGHTS = {
    "Turbidity": {
        "era5h_runoff_prior_24h_mm": 0.498,
        "era5h_runoff_prior_1h_mm": 0.406,
        "rain_gsmap_prior_24h_mm": 0.096,
    },
    "DO proxy": {
        "era5h_air_temp_prior_72h_C": 0.259,
        "era5h_wind_max_prior_168h_m_s": 0.215,
        "rain_gsmap_prior_720h_mm": 0.184,
        "rain_gsmap_prior_336h_mm": 0.179,
        "era5h_solar_prior_168h_MJ_m2": 0.104,
        "rainy_hours_gsmap_prior_336h": 0.058,
    },
}

context_audit = pd.read_csv(CONTEXT_AUDIT_PATH, low_memory=False)
context_audit["sat_date"] = pd.to_datetime(
    context_audit["sat_date"], errors="coerce"
).dt.strftime("%Y-%m-%d")
selected_context = (
    context_audit.loc[context_audit["sat_date"].isin(EXPECTED_DATES)]
    .sort_values("sat_date")
    .reset_index(drop=True)
)
if selected_context["sat_date"].tolist() != EXPECTED_DATES:
    raise AssertionError("Context audit does not contain both locked dates.")

# Prefer the preregistered z columns. Fall back to the model-data reference.
for feature in DYNAMIC_CONTEXT_FEATURES:
    z_name = f"z_{feature}"
    if z_name not in selected_context.columns:
        reference = pd.to_numeric(model_data[feature], errors="coerce").dropna()
        median = float(reference.median())
        scale = float(reference.quantile(0.75) - reference.quantile(0.25))
        if not np.isfinite(scale) or scale <= 0:
            scale = float(reference.std(ddof=0))
        if not np.isfinite(scale) or scale <= 0:
            scale = 1.0
        selected_context[z_name] = (
            pd.to_numeric(selected_context[feature], errors="coerce") - median
        ) / scale

context_rows = []
for feature in DYNAMIC_CONTEXT_FEATURES:
    value1 = float(selected_context.loc[0, feature])
    value2 = float(selected_context.loc[1, feature])
    z1 = float(selected_context.loc[0, f"z_{feature}"])
    z2 = float(selected_context.loc[1, f"z_{feature}"])
    weight_tur = SHAP_DYNAMIC_WEIGHTS["Turbidity"].get(feature, 0.0)
    weight_do = SHAP_DYNAMIC_WEIGHTS["DO proxy"].get(feature, 0.0)
    context_rows.append({
        "feature": feature,
        "label": CONTEXT_SHORT_LABELS[feature],
        "group": CONTEXT_GROUP[feature],
        "unit": CONTEXT_UNITS[feature],
        "date_1": EXPECTED_DATES[0],
        "date_2": EXPECTED_DATES[1],
        "value_date_1": value1,
        "value_date_2": value2,
        "raw_delta_date2_minus_date1": value2 - value1,
        "z_date_1": z1,
        "z_date_2": z2,
        "z_delta_date2_minus_date1": z2 - z1,
        "abs_z_delta": abs(z2 - z1),
        "turbidity_shap_weight": weight_tur,
        "do_proxy_shap_weight": weight_do,
        "turbidity_weighted_contrast": weight_tur * abs(z2 - z1),
        "do_proxy_weighted_contrast": weight_do * abs(z2 - z1),
    })
context_contrast = pd.DataFrame(context_rows).sort_values(
    ["abs_z_delta", "feature"], ascending=[False, True]
)
context_contrast.to_csv(
    TABLE_DIR / "figure2_context_contrast.csv",
    index=False, encoding="utf-8-sig",
)
display(context_contrast.round(4))


## 8. Define the satellite and model-context feature blocks

The block definitions follow the exact feature names stored in each
locked artifact. Features are not selected or tuned using these two
mapping dates. Static context features remain numerically unchanged
between dates and therefore contribute zero direct temporal change,
although they may still modify nonlinear model responses.


In [ ]:
S2_BLOCK_FEATURES = {
    "Turbidity": [
        "B1_median", "B2_median", "B3_median", "B4_median",
        "B5_median", "B6_median", "B7_median", "B8_median",
        "B8A_median", "B9_median", "B11_median", "B12_median",
        "NDTI_median", "NDWI_median",
    ],
    "SS": [
        "B1_median", "B2_median", "B3_median", "B4_median",
        "B5_median", "B6_median", "B7_median", "B8_median",
        "B8A_median", "B9_median", "B11_median", "B12_median",
        "NDTI_median", "NDWI_median", "NDWI1", "NDWI3",
        "B4_B3_ratio", "B8_B4_ratio", "NDWI2",
        "NIR_Blue_ratio", "NIR_Green_ratio", "NIR_Red_ratio",
        "Red_NIR_ND",
    ],
    "DO proxy": [
        "B2_median", "B3_median", "B4_median", "B5_median",
        "B6_median", "B7_median", "B8_median", "B8A_median",
        "B11_median", "B12_median",
    ],
}

feature_block_rows = []
CONTEXT_BLOCK_FEATURES = {}
for target in TARGET_ORDER:
    model_features = list(bundles[target]["feature_names"])
    s2_features = list(S2_BLOCK_FEATURES[target])
    missing = sorted(set(s2_features).difference(model_features))
    if missing:
        raise KeyError(f"{target}: S2 block contains non-model features: {missing}")
    context_features = [name for name in model_features if name not in s2_features]
    if not context_features:
        raise RuntimeError(f"{target}: model-context block is empty.")
    CONTEXT_BLOCK_FEATURES[target] = context_features
    for feature in model_features:
        feature_block_rows.append({
            "target": target,
            "feature": feature,
            "input_block": (
                "Satellite/scene" if feature in s2_features
                else "Model context"
            ),
            "is_dynamic_environmental_context": feature in DYNAMIC_CONTEXT_FEATURES,
        })

feature_block_audit = pd.DataFrame(feature_block_rows)
feature_block_audit.to_csv(
    TABLE_DIR / "figure2_feature_block_definition.csv",
    index=False, encoding="utf-8-sig",
)
display(
    feature_block_audit.groupby(["target", "input_block"], as_index=False)
    .agg(n_features=("feature", "size"))
)
for target in TARGET_ORDER:
    print(f"\n{target} model-context features:")
    print(CONTEXT_BLOCK_FEATURES[target])


## 9. Fast two-block counterfactual decomposition

For dates 1 and 2, let `S` denote the satellite/scene block and `C`
denote the model-context block. The deployed predictions are
`P11=f(S1,C1)` and `P22=f(S2,C2)`. Two additional crossed predictions
are evaluated: `P12=f(S1,C2)` and `P21=f(S2,C1)`.

The pixelwise components are:

\[
\Delta_{S}=	frac{1}{2}[(P_{21}-P_{11})+(P_{22}-P_{12})]
\]

\[
\Delta_{C}=	frac{1}{2}[(P_{12}-P_{11})+(P_{22}-P_{21})]
\]

so that `P22 - P11 = Delta_S + Delta_C`. The crossed configurations
are diagnostic model inputs and need not represent physically observed
states.


In [ ]:
TARGET_SLUG = {
    "Turbidity": "turbidity",
    "SS": "ss",
    "DO proxy": "do_proxy",
}
COMPONENT_LABELS = [
    "total_delta", "satellite_component",
    "context_component", "additivity_residual",
]
COUNTERFACTUAL_BANDS = [
    f"{TARGET_SLUG[target]}_{component}"
    for target in TARGET_ORDER
    for component in COMPONENT_LABELS
]
COUNTERFACTUAL_BAND_INDEX = {
    name: index + 1 for index, name in enumerate(COUNTERFACTUAL_BANDS)
}

def deployed_prediction(bundle, frame):
    _, values = exact_prediction(bundle, frame)
    return np.clip(np.asarray(values, dtype=float), 0.0, None)


def counterfactual_raster_path():
    return RASTER_DIR / (
        f"{date_jobs[0]['job_id']}_{date_jobs[1]['job_id']}_"
        "two_block_counterfactual.tif"
    )


def counterfactual_cache_valid(path):
    path = Path(path)
    if not path.exists():
        return False
    try:
        with rasterio.open(path) as source:
            return (
                source.count == len(COUNTERFACTUAL_BANDS)
                and list(source.descriptions) == COUNTERFACTUAL_BANDS
                and source.tags().get("status") == "complete"
            )
    except Exception:
        return False


def run_counterfactual_decomposition():
    output_path = counterfactual_raster_path()
    if counterfactual_cache_valid(output_path) and not OVERWRITE_COUNTERFACTUAL:
        print("Using cached counterfactual raster:", output_path)
        return output_path
    if output_path.exists():
        output_path.unlink()

    path1 = Path(date_jobs[0]["feature_path"])
    path2 = Path(date_jobs[1]["feature_path"])
    prediction_path1 = Path(prediction_paths[date_jobs[0]["job_id"]])
    prediction_path2 = Path(prediction_paths[date_jobs[1]["job_id"]])
    union_features = []
    for target in TARGET_ORDER:
        for feature in bundles[target]["feature_names"]:
            if feature not in union_features:
                union_features.append(feature)

    with (
        rasterio.open(path1) as source1,
        rasterio.open(path2) as source2,
        rasterio.open(prediction_path1) as prediction_source1,
        rasterio.open(prediction_path2) as prediction_source2,
    ):
        names1 = band_map(source1)
        names2 = band_map(source2)
        indexes1 = [names1[name] for name in union_features]
        indexes2 = [names2[name] for name in union_features]
        prediction_names1 = band_map(prediction_source1)
        prediction_names2 = band_map(prediction_source2)
        prediction_indexes1 = [
            prediction_names1[OUTPUT_NAME_BY_TARGET[target]]
            for target in TARGET_ORDER
        ]
        prediction_indexes2 = [
            prediction_names2[OUTPUT_NAME_BY_TARGET[target]]
            for target in TARGET_ORDER
        ]
        profile = source1.profile.copy()
        profile.update(
            driver="GTiff", count=len(COUNTERFACTUAL_BANDS),
            dtype="float32", nodata=NODATA, compress="deflate",
            predictor=3, BIGTIFF="IF_SAFER",
        )
        windows = list(iter_windows(source1.width, source1.height, BLOCK_SIZE))
        with rasterio.open(output_path, "w", **profile) as destination:
            for index, name in enumerate(COUNTERFACTUAL_BANDS, start=1):
                destination.set_band_description(index, name)
            destination.update_tags(
                workflow="figure2_fast_two_block_counterfactual",
                date_1=date_jobs[0]["sat_date"],
                date_2=date_jobs[1]["sat_date"],
                interpretation="model_based_not_causal",
                status="incomplete",
            )

            for window_number, window in enumerate(windows, start=1):
                row0 = int(window.row_off)
                row1 = row0 + int(window.height)
                col0 = int(window.col_off)
                col1 = col0 + int(window.width)
                common = strict_common_mask[row0:row1, col0:col1]
                selected = np.flatnonzero(common.reshape(-1))
                output = np.full(
                    (len(COUNTERFACTUAL_BANDS), int(window.height), int(window.width)),
                    NODATA, dtype=np.float32,
                )

                if selected.size:
                    raw1 = np.ma.asarray(
                        source1.read(indexes1, window=window, masked=True)
                    ).filled(np.nan).astype(float)
                    raw2 = np.ma.asarray(
                        source2.read(indexes2, window=window, masked=True)
                    ).filled(np.nan).astype(float)
                    observed1 = np.ma.asarray(
                        prediction_source1.read(
                            prediction_indexes1, window=window, masked=True
                        )
                    ).filled(np.nan).astype(float)
                    observed2 = np.ma.asarray(
                        prediction_source2.read(
                            prediction_indexes2, window=window, masked=True
                        )
                    ).filled(np.nan).astype(float)
                    values1 = {
                        name: raw1[index].reshape(-1)[selected]
                        for index, name in enumerate(union_features)
                    }
                    values2 = {
                        name: raw2[index].reshape(-1)[selected]
                        for index, name in enumerate(union_features)
                    }

                    for target in TARGET_ORDER:
                        bundle = bundles[target]
                        features = list(bundle["feature_names"])
                        context_features = CONTEXT_BLOCK_FEATURES[target]
                        frame1 = pd.DataFrame({name: values1[name] for name in features})
                        frame2 = pd.DataFrame({name: values2[name] for name in features})
                        frame12 = frame1.copy()
                        frame12.loc[:, context_features] = frame2[context_features].to_numpy()
                        frame21 = frame2.copy()
                        frame21.loc[:, context_features] = frame1[context_features].to_numpy()

                        target_position = TARGET_ORDER.index(target)
                        p11 = observed1[target_position].reshape(-1)[selected]
                        p12 = deployed_prediction(bundle, frame12)
                        p21 = deployed_prediction(bundle, frame21)
                        p22 = observed2[target_position].reshape(-1)[selected]

                        total = p22 - p11
                        satellite = 0.5 * ((p21 - p11) + (p22 - p12))
                        context = 0.5 * ((p12 - p11) + (p22 - p21))
                        residual = total - satellite - context
                        arrays = {
                            "total_delta": total,
                            "satellite_component": satellite,
                            "context_component": context,
                            "additivity_residual": residual,
                        }
                        for component, values in arrays.items():
                            band_name = f"{TARGET_SLUG[target]}_{component}"
                            target_output = output[
                                COUNTERFACTUAL_BAND_INDEX[band_name] - 1
                            ].reshape(-1)
                            target_output[selected] = values.astype(np.float32)

                destination.write(output, window=window)
                if (
                    window_number == 1
                    or window_number % 10 == 0
                    or window_number == len(windows)
                ):
                    print(
                        f"Counterfactual: window {window_number}/{len(windows)}"
                    )
            destination.update_tags(status="complete")
    print("Saved:", output_path)
    return output_path


COUNTERFACTUAL_RASTER_PATH = run_counterfactual_decomposition()


## 10. Summarize prediction and block-attribution changes


In [ ]:
component_arrays = {}
decomposition_rows = []
with rasterio.open(COUNTERFACTUAL_RASTER_PATH) as source:
    for target in TARGET_ORDER:
        slug = TARGET_SLUG[target]
        arrays = {
            component: read_named(source, f"{slug}_{component}")
            for component in COMPONENT_LABELS
        }
        component_arrays[target] = arrays
        total = arrays["total_delta"][strict_common_mask]
        satellite = arrays["satellite_component"][strict_common_mask]
        context = arrays["context_component"][strict_common_mask]
        stored_residual = arrays["additivity_residual"][strict_common_mask]
        residual = total - satellite - context
        denominator = np.sum(np.abs(satellite)) + np.sum(np.abs(context))
        satellite_share = (
            100 * np.sum(np.abs(satellite)) / denominator
            if denominator > 0 else np.nan
        )
        context_share = 100 - satellite_share if np.isfinite(satellite_share) else np.nan
        decomposition_rows.append({
            "target": target,
            "unit": TARGET_CONFIG[target]["unit"],
            "n_common_pixels": len(total),
            "total_delta_mean": np.mean(total),
            "total_delta_median": np.median(total),
            "total_delta_q25": np.quantile(total, 0.25),
            "total_delta_q75": np.quantile(total, 0.75),
            "satellite_component_mean": np.mean(satellite),
            "satellite_component_median": np.median(satellite),
            "context_component_mean": np.mean(context),
            "context_component_median": np.median(context),
            "satellite_absolute_share_pct": satellite_share,
            "context_absolute_share_pct": context_share,
            "satellite_dominant_pixel_pct": 100 * np.mean(np.abs(satellite) > np.abs(context)),
            "context_dominant_pixel_pct": 100 * np.mean(np.abs(context) > np.abs(satellite)),
            "positive_total_change_pct": 100 * np.mean(total > 0),
            "max_abs_additivity_residual": np.max(np.abs(residual)),
            "max_abs_stored_residual_band": np.max(np.abs(stored_residual)),
            "spearman_total_vs_satellite": spearmanr(total, satellite).statistic,
            "spearman_total_vs_context": spearmanr(total, context).statistic,
        })

counterfactual_summary = pd.DataFrame(decomposition_rows)
counterfactual_summary.to_csv(
    TABLE_DIR / "figure2_counterfactual_decomposition_summary.csv",
    index=False, encoding="utf-8-sig",
)
display(counterfactual_summary.round(4))


## 11. Main manuscript figure

The only figure generated by this notebook compares the observed-date
predictions on a common valid low-OOD water mask. Environmental-context
contrasts and the two-block decomposition are retained in CSV tables for
reporting rather than plotted as separate figures.


In [ ]:
def truncate_colormap(cmap_name, low=0.05, high=0.90, n=256):
    cmap = plt.get_cmap(cmap_name)
    return LinearSegmentedColormap.from_list(
        f"{cmap_name}_truncated", cmap(np.linspace(low, high, n))
    )


PREDICTION_CMAP = truncate_colormap("viridis", 0.05, 0.90)
DIFFERENCE_CMAP = plt.get_cmap("RdBu_r").copy()
DIFFERENCE_CMAP.set_bad((0, 0, 0, 0))

DISPLAY_UNITS = {
    "Turbidity": "NTU",
    "SS": r"mg L$^{-1}$",
    "DO proxy": r"mg L$^{-1}$",
}
ROW_LABELS = {
    "Turbidity": "Turbidity",
    "SS": "Suspended solids",
    "DO proxy": "DO proxy",
}
# Fixed, shared difference limits requested for the manuscript figure.
DIFFERENCE_LIMITS = {
    "Turbidity": 2.0,
    "SS": 0.16,
    "DO proxy": 0.25,
}


def crop_window_from_mask(mask, padding_fraction=0.025):
    rows, cols = np.where(mask)
    if not len(rows):
        raise ValueError("Cannot crop an empty mask.")
    row_min, row_max = rows.min(), rows.max() + 1
    col_min, col_max = cols.min(), cols.max() + 1
    pad_rows = max(2, int(round((row_max - row_min) * padding_fraction)))
    pad_cols = max(2, int(round((col_max - col_min) * padding_fraction)))
    row_min = max(0, row_min - pad_rows)
    row_max = min(mask.shape[0], row_max + pad_rows)
    col_min = max(0, col_min - pad_cols)
    col_max = min(mask.shape[1], col_max + pad_cols)
    return Window(col_min, row_min, col_max - col_min, row_max - row_min)


DISPLAY_WINDOW = crop_window_from_mask(strict_common_mask)
display_scale = min(
    1.0,
    PREVIEW_MAX_DIMENSION / max(DISPLAY_WINDOW.width, DISPLAY_WINDOW.height),
)
DISPLAY_SHAPE = (
    max(1, int(round(DISPLAY_WINDOW.height * display_scale))),
    max(1, int(round(DISPLAY_WINDOW.width * display_scale))),
)

with rasterio.open(Path(date_jobs[0]["feature_path"])) as scale_source:
    center_row = DISPLAY_WINDOW.row_off + DISPLAY_WINDOW.height / 2
    center_col = DISPLAY_WINDOW.col_off + DISPLAY_WINDOW.width / 2
    _, center_latitude = rasterio.transform.xy(
        scale_source.transform, center_row, center_col, offset="center"
    )
    crop_width_degrees = abs(scale_source.transform.a) * DISPLAY_WINDOW.width
for candidate_km in [10, 5, 2, 1]:
    longitude_degrees = candidate_km / (
        111.32 * max(np.cos(np.deg2rad(center_latitude)), 1e-6)
    )
    scale_bar_fraction = longitude_degrees / max(crop_width_degrees, 1e-9)
    if scale_bar_fraction <= 0.28:
        DISPLAY_SCALE_BAR_KM = candidate_km
        DISPLAY_SCALE_BAR_FRACTION = scale_bar_fraction
        break
else:
    DISPLAY_SCALE_BAR_KM = 1
    DISPLAY_SCALE_BAR_FRACTION = min(scale_bar_fraction, 0.28)


def read_window(source, band_name=None, indexes=None, resampling=Resampling.bilinear):
    if band_name is not None:
        indexes = band_map(source)[band_name]
    values = source.read(
        indexes,
        window=DISPLAY_WINDOW,
        out_shape=(
            (len(indexes), *DISPLAY_SHAPE)
            if isinstance(indexes, (list, tuple)) else DISPLAY_SHAPE
        ),
        resampling=resampling,
        masked=True,
    )
    return np.ma.asarray(values).filled(np.nan).astype(float)


strict_crop = strict_common_mask[
    int(DISPLAY_WINDOW.row_off):int(DISPLAY_WINDOW.row_off + DISPLAY_WINDOW.height),
    int(DISPLAY_WINDOW.col_off):int(DISPLAY_WINDOW.col_off + DISPLAY_WINDOW.width),
]
row_index = np.clip(
    np.floor(
        np.linspace(0, strict_crop.shape[0], DISPLAY_SHAPE[0], endpoint=False)
    ).astype(int),
    0, strict_crop.shape[0] - 1,
)
col_index = np.clip(
    np.floor(
        np.linspace(0, strict_crop.shape[1], DISPLAY_SHAPE[1], endpoint=False)
    ).astype(int),
    0, strict_crop.shape[1] - 1,
)
strict_preview = strict_crop[np.ix_(row_index, col_index)]


def grayscale_satellite_preview(job):
    with rasterio.open(job["rgb_path"]) as source:
        rgb = read_window(source, indexes=[1, 2, 3])
    rgb = np.moveaxis(rgb, 0, -1)
    finite = np.isfinite(rgb).all(axis=2)
    low, high = np.nanpercentile(rgb[finite], [2, 98]) if finite.any() else (0, 1)
    rgb = np.clip((rgb - low) / max(high - low, 1e-9), 0, 1)
    luminance = (
        0.2126 * rgb[..., 0]
        + 0.7152 * rgb[..., 1]
        + 0.0722 * rgb[..., 2]
    )
    # Light, desaturated satellite background across the full rectangle.
    luminance = np.clip(0.72 * luminance + 0.22, 0, 1)
    background = np.repeat(luminance[..., None], 3, axis=2)
    background[~finite] = 0.92
    return background


gray_previews = {
    job["job_id"]: grayscale_satellite_preview(job) for job in date_jobs
}
job1, job2 = date_jobs[0]["job_id"], date_jobs[1]["job_id"]

prediction_previews = {job1: {}, job2: {}}
for job in date_jobs:
    with rasterio.open(prediction_paths[job["job_id"]]) as source:
        for target in TARGET_ORDER:
            values = read_window(
                source,
                band_name=OUTPUT_NAME_BY_TARGET[target],
                resampling=Resampling.bilinear,
            )
            prediction_previews[job["job_id"]][target] = np.where(
                strict_preview, values, np.nan
            )

delta_previews = {}
with rasterio.open(COUNTERFACTUAL_RASTER_PATH) as source:
    for target in TARGET_ORDER:
        values = read_window(
            source,
            band_name=f"{TARGET_SLUG[target]}_total_delta",
            resampling=Resampling.bilinear,
        )
        delta_previews[target] = np.where(strict_preview, values, np.nan)


def finite_limits(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if not len(values):
        return 0.0, 1.0
    low, high = float(values.min()), float(values.max())
    if np.isclose(low, high):
        padding = max(abs(low) * 0.05, 0.1)
        return low - padding, high + padding
    return low, high


prediction_limits = {}
for target in TARGET_ORDER:
    pooled = np.concatenate([
        prediction_previews[job1][target][
            np.isfinite(prediction_previews[job1][target])
        ],
        prediction_previews[job2][target][
            np.isfinite(prediction_previews[job2][target])
        ],
    ])
    prediction_limits[target] = finite_limits(pooled)


def add_north_arrow(axis):
    axis.annotate(
        "N", xy=(0.94, 0.92), xytext=(0.94, 0.80),
        xycoords="axes fraction", textcoords="axes fraction",
        ha="center", va="center", fontsize=7,
        arrowprops=dict(arrowstyle="-|>", lw=0.75, color="black"),
    )


def add_scale_bar(axis):
    x0 = 0.06
    x1 = x0 + DISPLAY_SCALE_BAR_FRACTION
    axis.plot([x0, x1], [0.065, 0.065], transform=axis.transAxes, color="black", lw=1.35)
    axis.plot([x0, x0], [0.052, 0.078], transform=axis.transAxes, color="black", lw=0.7)
    axis.plot([x1, x1], [0.052, 0.078], transform=axis.transAxes, color="black", lw=0.7)
    axis.text(
        (x0 + x1) / 2, 0.088, f"{DISPLAY_SCALE_BAR_KM} km",
        transform=axis.transAxes, ha="center", va="bottom", fontsize=6.5,
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.72, pad=1),
    )


# Three map rows, a thin colorbar directly below each row, and a separate
# spacer before the next row. The tall page preserves geographic aspect.
figure = plt.figure(figsize=(7.2, 12.5))
grid = GridSpec(
    9, 3, figure=figure,
    height_ratios=[1, 0.026, 0.075, 1, 0.026, 0.075, 1, 0.026, 0.015],
    hspace=0.055, wspace=0.018,
)
column_titles = [
    "4 June 2024",
    "19 June 2024",
    "Δ prediction (19 June − 4 June)",
]
panel_letter = 0

for target_index, target in enumerate(TARGET_ORDER):
    map_row = 3 * target_index
    colorbar_row = map_row + 1
    axes = [figure.add_subplot(grid[map_row, column]) for column in range(3)]
    vmin, vmax = prediction_limits[target]
    difference_limit = DIFFERENCE_LIMITS[target]
    arrays = [
        prediction_previews[job1][target],
        prediction_previews[job2][target],
        delta_previews[target],
    ]
    backgrounds = [gray_previews[job1], gray_previews[job2], gray_previews[job2]]
    mappables = []

    for column, (axis, values, background) in enumerate(
        zip(axes, arrays, backgrounds)
    ):
        axis.imshow(background, interpolation="nearest")
        if column < 2:
            image = axis.imshow(
                values, cmap=PREDICTION_CMAP, vmin=vmin, vmax=vmax,
                alpha=np.where(np.isfinite(values), 0.97, 0.0),
                interpolation="nearest",
            )
        else:
            image = axis.imshow(
                values, cmap=DIFFERENCE_CMAP,
                norm=TwoSlopeNorm(
                    vmin=-difference_limit,
                    vcenter=0.0,
                    vmax=difference_limit,
                ),
                alpha=np.where(np.isfinite(values), 0.97, 0.0),
                interpolation="nearest",
            )
        axis.contour(
            strict_preview.astype(float), levels=[0.5],
            colors="#303030", linewidths=0.32,
        )
        axis.text(
            0.012, 0.985, f"({chr(97 + panel_letter)})",
            transform=axis.transAxes, ha="left", va="top",
            fontsize=8, fontweight="bold",
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.55, pad=0.7),
        )
        panel_letter += 1
        if target_index == 0:
            axis.set_title(
                column_titles[column], fontsize=8.5, fontweight="bold", pad=4
            )
        axis.set_axis_off()
        mappables.append(image)

    axes[0].text(
        -0.075, 0.5, ROW_LABELS[target],
        transform=axes[0].transAxes, rotation=90,
        va="center", ha="center", fontsize=8.5, fontweight="bold",
    )
    if target_index == 0:
        add_north_arrow(axes[2])
    if target_index == 2:
        add_scale_bar(axes[0])

    prediction_cax = figure.add_subplot(grid[colorbar_row, 0:2])
    prediction_cb = figure.colorbar(
        mappables[0], cax=prediction_cax, orientation="horizontal"
    )
    prediction_cb.locator = MaxNLocator(nbins=4)
    prediction_cb.update_ticks()
    prediction_cb.ax.tick_params(labelsize=7, length=2, width=0.45, pad=1.5)
    prediction_cb.outline.set_linewidth(0.5)
    prediction_cb.set_label(
        f"{ROW_LABELS[target]} ({DISPLAY_UNITS[target]})",
        fontsize=7.5, labelpad=2,
    )

    difference_cax = figure.add_subplot(grid[colorbar_row, 2])
    difference_cb = figure.colorbar(
        mappables[2], cax=difference_cax,
        orientation="horizontal", extend="both", extendfrac=0.045,
    )
    difference_cb.set_ticks([-difference_limit, 0.0, difference_limit])
    decimals = 2 if difference_limit < 1 else 0
    difference_cb.ax.xaxis.set_major_formatter(
        FuncFormatter(
            lambda value, _: (
                "0" if np.isclose(value, 0)
                else f"{value:+.{decimals}f}"
            )
        )
    )
    difference_cb.ax.tick_params(labelsize=7, length=2, width=0.45, pad=1.5)
    difference_cb.outline.set_linewidth(0.5)
    difference_cb.set_label(
        rf"$\Delta$ {ROW_LABELS[target]} ({DISPLAY_UNITS[target]})",
        fontsize=7.5, labelpad=2,
    )
    median_delta = counterfactual_summary.loc[
        counterfactual_summary["target"].eq(target),
        "total_delta_median",
    ].iloc[0]
    difference_cax.text(
        0.5, 1.72,
        rf"Median $\Delta$ = {median_delta:+.2f} {DISPLAY_UNITS[target]}",
        transform=difference_cax.transAxes,
        ha="center", va="bottom", fontsize=7.2,
    )

figure.subplots_adjust(
    left=0.055, right=0.995, top=0.975, bottom=0.025,
)
MAIN_FIGURE_PNG = FIGURE_DIR / "figure2_main_prediction_change.png"
MAIN_FIGURE_PDF = FIGURE_DIR / "figure2_main_prediction_change.pdf"
figure.savefig(MAIN_FIGURE_PNG, dpi=300, bbox_inches="tight")
figure.savefig(MAIN_FIGURE_PDF, bbox_inches="tight")
plt.show()
print("Main manuscript figure:", MAIN_FIGURE_PNG)


## 12. Completion gate and reporting guardrails

The notebook produces one manuscript figure. Contextual-condition values,
feature-block definitions, and counterfactual decomposition results are
retained as tables. The decomposition describes fitted-model behavior and
must not be interpreted as a causal environmental effect.


In [ ]:
expected_outputs = [
    TABLE_DIR / "figure2_locked_model_inventory.csv",
    TABLE_DIR / "figure2_training_split_audit.csv",
    TABLE_DIR / "figure2_input_grid_audit.csv",
    TABLE_DIR / "figure2_common_mask_audit.csv",
    TABLE_DIR / "figure2_context_contrast.csv",
    TABLE_DIR / "figure2_prediction_change_summary.csv",
    TABLE_DIR / "figure2_feature_block_definition.csv",
    TABLE_DIR / "figure2_counterfactual_decomposition_summary.csv",
    FIGURE_DIR / "figure2_main_prediction_change.png",
    FIGURE_DIR / "figure2_main_prediction_change.pdf",
    COUNTERFACTUAL_RASTER_PATH,
]

checks = pd.DataFrame([
    {
        "check": "two_locked_dates",
        "passed": jobs["sat_date"].tolist() == EXPECTED_DATES,
        "detail": ", ".join(jobs["sat_date"]),
    },
    {
        "check": "selection_independent_of_predictions",
        "passed": not selected_manifest["selection_used_predictions"].astype(str).str.lower().isin(["true", "1"]).any(),
        "detail": selected_manifest["selection_source"].iloc[0],
    },
    {
        "check": "same_grid_and_schema",
        "passed": grid_audit[["feature_grid_matches_reference", "rgb_grid_matches_feature", "model_features_complete"]].all().all(),
        "detail": "two 62-band feature rasters and two RGB rasters",
    },
    {
        "check": "strict_common_mask_nonempty",
        "passed": int(strict_common_mask.sum()) > 0,
        "detail": f"{int(strict_common_mask.sum()):,} pixels",
    },
    {
        "check": "exact_two_block_additivity",
        "passed": counterfactual_summary["max_abs_additivity_residual"].max() < 1e-4,
        "detail": f"max residual={counterfactual_summary['max_abs_additivity_residual'].max():.3g}",
    },
    {
        "check": "outputs_complete",
        "passed": all(path.exists() for path in expected_outputs),
        "detail": f"{sum(path.exists() for path in expected_outputs)}/{len(expected_outputs)} files",
    },
])
checks.to_csv(
    TABLE_DIR / "figure2_fast_completion_gate.csv",
    index=False, encoding="utf-8-sig",
)
display(checks)
FIGURE_2_FAST_READY = bool(checks["passed"].all())
print("FIGURE_2_FAST_READY =", FIGURE_2_FAST_READY)
print("\nReporting guardrails:")
print("- Use 'model-context contribution', not causal contribution.")
print("- The crossed states are model diagnostics, not observed environmental states.")
print("- DO is reported as an indirect DO proxy.")


In [ ]:
# Optional: choose your own private destination and uncomment to persist results.
# import shutil
# destination = Path("/content/drive/MyDrive/wqr_my_results")
# shutil.copytree(OUTPUT_ROOT, destination, dirs_exist_ok=True)
print("Results for this run:", OUTPUT_ROOT)
